In [ ]:
!pip install -q scikit-learn
!pip install tensorflow-estimator



In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from six.moves import urllib

import tensorflow.compat.v2.feature_column as fc

import tensorflow as tf

In [ ]:
#Block 1
dftrain = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/train.csv') # training data
dfeval = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/eval.csv') # testing data
y_train = dftrain.pop('survived')
y_eval = dfeval.pop('survived')

categorical_columns = ['sex','n_siblings_spouses','parch','class','deck','embark_town','alone']
numeric_columns = ['age','fare']

#Have to create feature columns for linear regression
feature_columns = []
for feature_name in categorical_columns:
  vocabulary = dftrain[feature_name].astype(str).unique() #получается feature name это 'sex','n_siblings_spouses','parch' и тд, и из этого он забирает именно УНИКАЛЬНЫЕ значение ( как лист всех значений)
  lookup = tf.keras.layers.StringLookup(vocabulary=vocabulary, output_mode='one_hot',)
  feature_columns.append((feature_name, lookup))
#обьяснение строки отдельно
#model needs to know is it categorical on numerical column, but in newest version of TF and keras its gonna be different
for feature_name in numeric_columns:
    normalizer = tf.keras.layers.Normalization(axis=None,)
    values = dftrain[feature_name].values.astype(np.float32).reshape(-1, 1)
    normalizer.adapt(values)
    feature_columns.append((feature_name, normalizer))

print(feature_columns)

[('sex', <StringLookup name=string_lookup_17, built=False>), ('n_siblings_spouses', <StringLookup name=string_lookup_18, built=False>), ('parch', <StringLookup name=string_lookup_19, built=False>), ('class', <StringLookup name=string_lookup_20, built=False>), ('deck', <StringLookup name=string_lookup_21, built=False>), ('embark_town', <StringLookup name=string_lookup_22, built=False>), ('alone', <StringLookup name=string_lookup_23, built=False>), ('age', <Normalization name=normalization_6, built=True>), ('fare', <Normalization name=normalization_7, built=True>)]


In [ ]:
#Block 2
def make_input_fn(data_df, label_df, num_epochs=10, shuffle=True, batch_size=32): #input function, and inside def returns the function object  #data df is pandas dataframe . label df is those y_eval y_train etc.
  def input_function():  # inner function, this will be returned
    data_df_copy = data_df.copy()
    for col in categorical_columns:
        data_df_copy[col] = data_df_copy[col].astype(str)
    ds = tf.data.Dataset.from_tensor_slices((dict(data_df_copy), label_df))
    if shuffle:
      ds = ds.shuffle(1000)  # randomize order of data
    ds = ds.batch(batch_size).repeat(num_epochs)  # split dataset into batches of 32 and repeat process for number of epochs
    return ds  # return a batch of the dataset
  return input_function  # return a function object for use

train_input_fn = make_input_fn(dftrain, y_train)  # here we will call the input_function that was returned to us to get a dataset object we can feed to the model
eval_input_fn = make_input_fn(dfeval, y_eval, num_epochs=1, shuffle=False) #epochs not the same. use testing data

train_ds = train_input_fn()  # получаем датасет из input_fn
eval_ds = eval_input_fn()
# создаем модель на основе feature_columns, подготовленных с Keras слоями
inputs = {}
encoded_features = []

for name, feature_layer in feature_columns:
    input_ = tf.keras.Input(shape=(1,), name=name, dtype=tf.string if isinstance(feature_layer, tf.keras.layers.StringLookup) else tf.float32)
    encoded = feature_layer(input_)
    inputs[name] = input_
    encoded_features.append(encoded)

concatenated = tf.keras.layers.concatenate(encoded_features)
output = tf.keras.layers.Dense(1, activation='sigmoid')(concatenated)

model = tf.keras.Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# training the model
model.fit(train_ds, verbose=0)

result = model.evaluate(eval_ds, return_dict=True)

clear_output()  # clears console output
print(result['accuracy'])
print(result)
#полное обьяснение в доксе

0.7348484992980957
{'accuracy': 0.7348484992980957, 'loss': 0.5374009609222412}


In [ ]:
#Create classifications for probabiliteis etc

import numpy as np

def predict_with_details(model, dataset):
    raw_preds = model.predict(dataset)
    results = []
    for p in raw_preds:
        p_val = float(p[0])  # т.к. model.predict возвращает [[0.8], [0.3], ...]
        prob = [1 - p_val, p_val]
        logit = np.log(p_val / (1 - p_val + 1e-7))  # логит, добавлен ε чтобы не делить на 0
        class_id = int(p_val >= 0.5)
        results.append({
            'probabilities': prob,
            'logits': logit,
            'class_ids': [class_id]
        })
    return results


In [ ]:
#Block 3
predictions = predict_with_details(model,eval_ds) #represents each dictionary
print(dfeval.loc[2]) #вывод информацию о пассажире под индексом 3
print(y_eval.loc[2]) #реальный ответ, выжил ли пассажир с индексом 3
print(predictions[2]['probabilities'][1]) #Это — предсказанная моделью вероятность того, что пассажир под индексом 3 выжил (class 1).

#predictions[3] — предсказание для пассажира №3
#['probabilities'] — массив двух значений: [P(класс 0), P(класс 1)]
#['probabilities'][1] — это вероятность класса 1 (то есть "выжил")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
sex                        female
age                          58.0
n_siblings_spouses              0
parch                           0
fare                        26.55
class                       First
deck                            C
embark_town           Southampton
alone                           y
Name: 2, dtype: object
1
0.4389817416667938
